# HEAL-CITY — Exploratory Data Analysis (EDA)

This notebook contains the complete **Exploratory Data Analysis (EDA)** phase for the HEAL-CITY smart city healthcare dataset. It aims to understand demographic and healthcare demand-capacity relationships across Surabaya's 31 Kecamatan.

## 01. Import Library
We import the required libraries for statistical analysis and visualizations, including interactive Plotly charts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

pd.set_option('display.max_columns', None)

## 02. Load Dataset
Load the aggregated master dataset `master_heal_city.csv` generated by the preprocessing pipeline.

In [ ]:
file_path = "../dataset/processed/master_heal_city.csv"
df = pd.read_csv(file_path)

# Add aliased/calculated columns matching spec terminology
df["nakes_per_1000"] = df["workforce_ratio"]
df["perawat_per_1000"] = df["jumlah_perawat"] / df["jumlah_penduduk"]
df["faskes_per_100k"] = df["facility_ratio"]
df["puskesmas_per_100k"] = (df["jumlah_puskesmas"] / df["jumlah_penduduk"]) * 100.0
df["visits_per_1000"] = df["service_pressure"]
df["disease_per_1000"] = df["disease_burden"]
df["beds_per_1000"] = df["bed_ratio"]

print("Master dataset dimensions:", df.shape)
display(df.head())

## 03. Dataset Overview
Let's review the basic structure, column names, data types, missing values, and duplicate rows.

In [ ]:
print("Jumlah baris:", df.shape[0])
print("Jumlah kolom:", df.shape[1])
print("\nData info summary:")
df.info()

## 04. Data Quality Check
Verify missing values rate per variable to ensure completeness of the aggregated indicators.

In [ ]:
missing = df.isna().mean().mul(100).sort_values(ascending=False)
print("Persentase Missing Value:")
print(missing)

plt.figure(figsize=(12, 5))
missing.plot(kind="bar", color="#3a86c8")
plt.title("Persentase Missing Value per Variabel")
plt.ylabel("Persentase (%)")
plt.xlabel("Variabel")
plt.tight_layout()
plt.show()

## 05. Population Analysis
Analyze demographic distribution across all 31 Kecamatan.

In [ ]:
print("Statistik Deskriptif Populasi (dalam Ribu):")
display(df["jumlah_penduduk"].describe())

fig = px.bar(
    df.sort_values("jumlah_penduduk", ascending=False),
    x="kecamatan",
    y="jumlah_penduduk",
    title="Jumlah Penduduk per Kecamatan (Ribu)",
    color="jumlah_penduduk",
    color_continuous_scale="Viridis"
)
fig.show()

## 06. Healthcare Workforce Analysis
Check total health workforce and population-scaled workforce ratio per 1,000 residents.

In [ ]:
print("Statistik Deskriptif Tenaga Kesehatan per 1.000 Penduduk:")
display(df["nakes_per_1000"].describe())

fig1 = px.bar(
    df.sort_values("total_tenaga_kesehatan", ascending=False),
    x="kecamatan",
    y="total_tenaga_kesehatan",
    title="Jumlah Absolut Tenaga Kesehatan per Kecamatan"
)
fig1.show()

fig2 = px.bar(
    df.sort_values("nakes_per_1000", ascending=False),
    x="kecamatan",
    y="nakes_per_1000",
    title="Tenaga Kesehatan per 1.000 Penduduk (Nakes Ratio)"
)
fig2.show()

## 07. Healthcare Facility Analysis
Analyze facility count distribution and normalized ratio per 100,000 residents.

In [ ]:
fig1 = px.bar(
    df.sort_values("total_faskes", ascending=False),
    x="kecamatan",
    y="total_faskes",
    title="Jumlah Fasilitas Kesehatan per Kecamatan"
)
fig1.show()

fig2 = px.bar(
    df.sort_values("faskes_per_100k", ascending=False),
    x="kecamatan",
    y="faskes_per_100k",
    title="Fasilitas Kesehatan per 100.000 Penduduk (Faskes Ratio)"
)
fig2.show()

## 08. Healthcare Visit Analysis
Explore total visit volumes and service pressure ratio (visits per 1,000 population).

In [ ]:
print("Statistik Deskriptif Kunjungan Puskesmas:")
display(df["total_kunjungan"].describe())

fig = px.bar(
    df.sort_values("visits_per_1000", ascending=False),
    x="kecamatan",
    y="visits_per_1000",
    title="Visits per 1.000 Penduduk (Service Pressure)"
)
fig.show()

## 09. Disease Burden Analysis
Examine the prevalence of cases per 1,000 population (Disease Burden).

In [ ]:
print("Statistik Deskriptif Beban Penyakit:")
display(df["total_kasus_penyakit"].describe())

fig = px.bar(
    df.sort_values("disease_per_1000", ascending=False),
    x="kecamatan",
    y="disease_per_1000",
    title="Kasus Penyakit per 1.000 Penduduk (Disease Burden)"
)
fig.show()

## 10. Healthcare Capacity Analysis
Check bed capacity normalized per 1,000 population (Bed Ratio).

In [ ]:
print("Statistik Deskriptif Kapasitas Tempat Tidur:")
display(df["total_tempat_tidur"].describe())

fig = px.bar(
    df.sort_values("beds_per_1000", ascending=False),
    x="kecamatan",
    y="beds_per_1000",
    title="Tempat Tidur per 1.000 Penduduk (Bed Ratio)"
)
fig.show()

## 11. Demand vs Capacity Analysis
Compare workload demands with capacity variables to detect visual mismatches.

In [ ]:
fig1 = px.scatter(
    df, x="nakes_per_1000", y="visits_per_1000", text="kecamatan",
    title="Healthcare Demand (Visits) vs Workforce Capacity (Nakes)"
)
fig1.show()

fig2 = px.scatter(
    df, x="faskes_per_100k", y="visits_per_1000", text="kecamatan",
    title="Healthcare Demand (Visits) vs Facility Capacity (Faskes)"
)
fig2.show()

fig3 = px.scatter(
    df, x="nakes_per_1000", y="disease_per_1000", text="kecamatan",
    title="Disease Burden vs Healthcare Workforce"
)
fig3.show()

## 12. Correlation Analysis
Calculate Pearson correlation coefficients and draw the correlation heatmap.

In [ ]:
corr_columns = [
    "jumlah_penduduk", "total_kunjungan", "total_tenaga_kesehatan", "total_faskes",
    "total_tempat_tidur", "total_kasus_penyakit", "visits_per_1000", "nakes_per_1000",
    "faskes_per_100k", "beds_per_1000", "disease_per_1000"
]
corr = df[corr_columns].corr()

fig = px.imshow(
    corr, text_auto=True, aspect="auto",
    title="Correlation Matrix HEAL-CITY Variables"
)
fig.show()

## 13. Outlier Analysis
Use Interquartile Range (IQR) to identify statistical outliers in core variables.

In [ ]:
def detect_outlier_iqr(df, column):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return df[(df[column] < lower) | (df[column] > upper)]

print("Outliers in jumlah_penduduk:")
display(detect_outlier_iqr(df, "jumlah_penduduk")[["kecamatan", "jumlah_penduduk"]])

print("\nOutliers in visits_per_1000:")
display(detect_outlier_iqr(df, "visits_per_1000")[["kecamatan", "visits_per_1000"]])

## 14. Feature Relationship Analysis
Plot the relationship between absolute population size and absolute visits, displaying an OLS regression trendline.

In [ ]:
fig = px.scatter(
    df, x="jumlah_penduduk", y="total_kunjungan", text="kecamatan",
    trendline="ols", title="Hubungan Antara Jumlah Penduduk dan Total Kunjungan"
)
fig.show()

## 15. Preliminary Healthcare Gap Analysis
Identify early workforce mismatch flags (Kecamatan with above-median service pressure and below-median nakes ratio).

In [ ]:
high_pressure = (df["visits_per_1000"] >= df["visits_per_1000"].median())
low_workforce = (df["nakes_per_1000"] <= df["nakes_per_1000"].median())
df["workforce_mismatch_flag"] = (high_pressure & low_workforce)

print("Kecamatan dengan Indikasi Mismatch Workforce (High Pressure + Low Nakes Ratio):")
display(df[df["workforce_mismatch_flag"]][["kecamatan", "visits_per_1000", "nakes_per_1000"]])

## 16. Feature Selection
We inspect the feature selection candidates report generated from our EDA.

In [ ]:
candidates = pd.read_csv("../outputs/eda/feature_candidates.csv")
display(candidates)

## 17. EDA Summary
Summary of core findings from the EDA.

In [ ]:
summary_df = pd.read_csv("../outputs/eda/eda_summary.csv")
display(summary_df)